# NB04 – Generator: Stable Diffusion XL  (`sdxl`)

**Run on its own Kaggle account, in parallel with the other generators.**

_1024×1024, 8 steps (plenty for AI signal) | `.to('cuda')` with VAE slicing_

### Prerequisites
- NB00, NB01, NB02 finished (repo, library, config, real images, captions exist).
- Internet **ON**, GPU **ON** (T4).
- `HF_TOKEN` secret set on this account.

### Parallel-safe
Writes only to `data/sdxl/`. Seed = `hash(MASTER_SEED, sdxl, source_real_id)`
— crashes/retries reproduce byte-identical images. Commits every 20 min. Stops at 10 K.

In [1]:
import importlib.util, sys, subprocess, os

# ── 1. cache: point HF to /kaggle/working so weights survive session restarts ──
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
print("HF_HOME:", os.environ["HF_HOME"])

# ── 2. PIL._Ink fix – run BEFORE any import touching torchvision/transformers ──
import PIL._typing
if not hasattr(PIL._typing, "_Ink"):
    from typing import Union
    PIL._typing._Ink = Union[int, float, str, bytes, tuple]
    for _k in list(sys.modules):
        if _k in ("PIL.ImageText","PIL.ImageDraw") or _k.startswith("torchvision"):
            del sys.modules[_k]
    print("patched PIL._typing._Ink")

# ── 3. install diffusers if missing/outdated – never -U on unrelated packages ──
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
_need = []
try:
    import diffusers as _d
    _ver = tuple(int(x) for x in _d.__version__.split(".")[:2])
    if _ver < (0, 30): _need.append("diffusers>=0.30")
except ImportError:
    _need.append("diffusers>=0.30")
for _m, _p in [("safetensors","safetensors"),
               ("sentencepiece","sentencepiece"),("protobuf","protobuf")]:
    if importlib.util.find_spec(_m) is None: _need.append(_p)
if _need: _pip(*_need); print("installed:", _need)
else: print("all deps present – no upgrades made")
import diffusers, torch
print(f"diffusers {diffusers.__version__} | torch {torch.__version__} | cuda: {torch.cuda.is_available()}")

HF_HOME: /kaggle/working/hf_cache
patched PIL._typing._Ink
installed: ['protobuf']
diffusers 0.36.0 | torch 2.10.0+cu128 | cuda: True


In [2]:
REPO_ID = "Shanmuk4622/ai-detection-dataset-v2"
MODEL   = "sdxl"   # ← the only line that differs between NB03–NB08

import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
MASTER_SEED = cfg.get("master_seed", 42)   # change master_seed in NB00 to reseed everything
print(f"library {C.PIPELINE_VERSION} loaded | MASTER_SEED={MASTER_SEED}")
assert MODEL in cfg["generators"], f"{MODEL} not in config"
g = cfg["generators"][MODEL]
print("spec:", g)

ai_detect_common.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

library 1.2 loaded | MASTER_SEED=42
spec: {'model_id': 'stabilityai/stable-diffusion-xl-base-1.0', 'pipeline': 'sdxl', 'native': 1024, 'steps': 8, 'guidance': 7.0}


In [3]:
import torch

# Import only the specific classes needed — AutoPipelineForText2Image cannot be imported
# because diffusers' auto_pipeline loads HunyuanDiT which requires MT5Tokenizer, which
# has been removed from newer transformers exports.
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionXLPipeline,
    FluxPipeline,
    KandinskyV22PriorPipeline,
    KandinskyV22Pipeline,
    PixArtSigmaPipeline,
    StableCascadePriorPipeline,
    StableCascadeDecoderPipeline,
)

DTYPE = torch.float16

def build_generator(model_key, g):
    mid    = g["model_id"]
    native = g["native"]
    steps  = g["steps"]
    guid   = g["guidance"]

    if model_key == "sd15":
        pipe = StableDiffusionPipeline.from_pretrained(
            mid, torch_dtype=DTYPE, safety_checker=None, requires_safety_checker=False)
        pipe.to("cuda")
        try: pipe.enable_vae_slicing()
        except Exception: pass
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt, height=native, width=native,
                        num_inference_steps=steps, guidance_scale=guid, generator=gn).images[0]
        return gen

    if model_key == "sdxl":
        pipe = StableDiffusionXLPipeline.from_pretrained(
            mid, torch_dtype=DTYPE, use_safetensors=True, add_watermarker=False)
        pipe.to("cuda")
        try: pipe.enable_vae_slicing()
        except Exception: pass
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt=prompt, height=native, width=native,
                        num_inference_steps=steps, guidance_scale=guid, generator=gn).images[0]
        return gen

    if model_key == "flux_schnell":
        pipe = FluxPipeline.from_pretrained(mid, torch_dtype=DTYPE)
        pipe.enable_model_cpu_offload()
        try: pipe.vae.enable_slicing(); pipe.vae.enable_tiling()
        except Exception: pass
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt=prompt, height=native, width=native,
                        num_inference_steps=steps, guidance_scale=guid,
                        max_sequence_length=256, generator=gn).images[0]
        return gen

    if model_key == "kandinsky22":
        # Explicit prior + decoder — avoids AutoPipeline and its broken HunyuanDiT import.
        prior = KandinskyV22PriorPipeline.from_pretrained(
            "kandinsky-community/kandinsky-2-2-prior", torch_dtype=DTYPE)
        prior.to("cuda")
        pipe = KandinskyV22Pipeline.from_pretrained(mid, torch_dtype=DTYPE)
        pipe.to("cuda")
        prior.set_progress_bar_config(disable=True)
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            prior_out = prior(prompt, num_inference_steps=steps,
                              guidance_scale=guid, generator=gn)
            return pipe(image_embeds=prior_out.image_embeds,
                        negative_image_embeds=prior_out.negative_image_embeds,
                        height=native, width=native,
                        num_inference_steps=steps, generator=gn).images[0]
        return gen

    if model_key == "pixart_sigma":
        pipe = PixArtSigmaPipeline.from_pretrained(mid, torch_dtype=DTYPE, use_safetensors=True)
        pipe.to("cuda")
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt=prompt, height=native, width=native,
                        num_inference_steps=steps, guidance_scale=guid, generator=gn).images[0]
        return gen

    if model_key == "sd_cascade":
        prior   = StableCascadePriorPipeline.from_pretrained(g["prior_id"], torch_dtype=DTYPE)
        decoder = StableCascadeDecoderPipeline.from_pretrained(mid, torch_dtype=DTYPE)
        prior.enable_model_cpu_offload()
        decoder.enable_model_cpu_offload()
        prior.set_progress_bar_config(disable=True)
        decoder.set_progress_bar_config(disable=True)
        p_steps, d_steps = steps[0], steps[1]
        p_guid,  d_guid  = guid[0],  guid[1]
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            po = prior(prompt=prompt, height=native, width=native, negative_prompt="",
                       guidance_scale=p_guid, num_inference_steps=p_steps,
                       num_images_per_prompt=1, generator=gn)
            return decoder(image_embeddings=po.image_embeddings, prompt=prompt,
                           negative_prompt="", guidance_scale=d_guid,
                           num_inference_steps=d_steps, output_type="pil", generator=gn).images[0]
        return gen

    raise ValueError(f"unknown model_key: {model_key}")

generate = build_generator(MODEL, g)
print("pipeline ready for", MODEL)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

pipeline ready for sdxl


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionXLPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [ ]:
import gc, time, torch
api      = C.HfApi(token=TOKEN)
TARGET   = cfg["target_per_generator"]          # 10_000
native   = g["native"]
g_steps  = g["steps"][0]  if isinstance(g["steps"],  list) else g["steps"]
g_guid   = g["guidance"][0] if isinstance(g["guidance"], list) else g["guidance"]

# load FROZEN captions (written by NB02) → {source_real_id: caption}
captions = {}
for f in C.list_shards(REPO_ID, "captions/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(local, columns=["source_real_id", "caption"])
    for sid, cap in zip(t.column("source_real_id").to_pylist(), t.column("caption").to_pylist()):
        captions[sid] = cap
assert captions, "No captions found – run NB02 first."
print(f"captions loaded: {len(captions)}")

# resume – authoritative count comes from the shards, not progress.json
done = C.reconcile_ids(REPO_ID, f"data/{MODEL}/", TOKEN)
todo = sorted(set(captions) - done)
print(f"done={len(done)} | remaining={len(todo)} | target={TARGET}")

writer = C.ShardWriter(api, REPO_ID, f"data/{MODEL}/",
                       commit_interval=cfg["commit_interval_seconds"],
                       max_rows=cfg["batch_size"], token=TOKEN)
accepted, failed = 0, 0
t0 = time.time()
try:
    for sid in todo:
        if len(done) + accepted >= TARGET:
            print("target reached."); break
        prompt = captions[sid]
        seed   = C.make_seed(MODEL, sid, MASTER_SEED)   # master_seed from config
        try:
            img = generate(prompt, seed)
            png = C.canonical_preprocess(img)
        except Exception as e:
            failed += 1
            if failed <= 10 or failed % 100 == 0:
                print(f"  gen failed {sid}: {type(e).__name__}: {e}")
            continue
        num = str(sid).split("_")[-1]
        row = C.empty_row()
        row.update(dict(image_id=f"{MODEL}_{num}", source_real_id=sid, label=1,
                        generator=MODEL, source_dataset=MODEL, prompt=prompt, image=png,
                        width=C.TARGET, height=C.TARGET, orig_width=native, orig_height=native,
                        gen_model_id=g["model_id"], gen_steps=int(g_steps),
                        gen_guidance=float(g_guid), gen_native_res=int(native),
                        seed=int(seed), caption_model=cfg["caption_model"],
                        pipeline_version=C.PIPELINE_VERSION,
                        sha256=C.sha256_bytes(png), created_utc=C.now_utc()))
        writer.add(row); accepted += 1
        if accepted % 100 == 0:
            elapsed = time.time() - t0
            rate    = accepted / elapsed
            eta_h   = ((TARGET - len(done) - accepted) / rate / 3600) if rate > 0 else 0
            print(f"  {MODEL}: {len(done)+accepted}/{TARGET} | "
                  f"{rate:.2f} img/s | ETA {eta_h:.1f} h | failed {failed}")
            gc.collect(); torch.cuda.empty_cache()
        writer.maybe_flush()
finally:
    writer.close()
total = len(done) + accepted
print(f"done. accepted this run: {accepted} | failed: {failed} | total: {total}/{TARGET}")

captions/captions-906881cf-00000.parquet:   0%|          | 0.00/23.2k [00:00<?, ?B/s]

captions/captions-906881cf-00001.parquet:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

captions/captions-906881cf-00002.parquet:   0%|          | 0.00/23.3k [00:00<?, ?B/s]

captions/captions-906881cf-00003.parquet:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

captions/captions-906881cf-00004.parquet:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

...

New Data Upload: |          |  0.00B /  0.00B            

  committed 127 rows (1401 total) -> data/sdxl/sdxl-eedb9c47-00010.parquet
  sdxl: 1500/10000 | 0.11 img/s | ETA 22.4 h | failed 0


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  committed 128 rows (1529 total) -> data/sdxl/sdxl-eedb9c47-00011.parquet


## Self-check

In [ ]:
n = C.count_rows(REPO_ID, f"data/{MODEL}/", TOKEN)
print(f"{MODEL}: {n} rows  (target {cfg['target_per_generator']})")
import random
shards = C.list_shards(REPO_ID, f"data/{MODEL}/", TOKEN)
if shards:
    loc = C.hf_hub_download(REPO_ID, shards[-1], repo_type="dataset", token=TOKEN)
    t   = C.pq.read_table(loc, columns=["image","image_id","source_real_id","label","seed"])
    j   = random.randrange(t.num_rows)
    ok, why = C.png_is_canonical(t.column("image")[j].as_py())
    print("sample:", t.column("image_id")[j].as_py(),
          "←", t.column("source_real_id")[j].as_py(),
          "| label", t.column("label")[j].as_py(),
          "| seed", t.column("seed")[j].as_py(),
          "| canonical", ok, why)
print("RESULT:", "looks correct" if n > 0 else "no rows yet")